In [1]:
from sktime.classification.feature_based import SummaryClassifier
from sktime.classification.feature_based import Catch22Classifier
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from sktime.classification.deep_learning.cnn import CNNClassifierTorch
from sktime.classification.deep_learning.resnet import ResNetClassifier
from sktime.classification.deep_learning.inceptiontime import InceptionTimeClassifierTorch
from sktime.classification.deep_learning.lstmfcn import LSTMFCNClassifierTorch
from sklearn.metrics import classification_report

import pandas as pd
import numpy as np
from datetime import datetime

def debug(msg: str) -> None:
    """Log messages with timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

In [ ]:
df_tmp = catalog.load("training.df_splitted")
df_tmp["real_data_path"] = "../data/03_primary/time_series_data/test/" + df_tmp["ism330dhcx_acc"]

all_labels = sorted(df_tmp["anomaly_label"].dropna().unique())
all_domains = sorted(df_tmp["domain_shift_op"].dropna().unique())
label_map = {label: idx for idx, label in enumerate(all_labels)}
df_tmp["anomaly_label_id"] = df_tmp["anomaly_label"].map(label_map)

X_train, y_train, X_test, y_test = [], [], [], []
for i in range(len(df_tmp)):
    ts = pd.read_parquet(df_tmp.loc[i, "real_data_path"])[["A_x [g]", "A_y [g]", "A_z [g]"]]
    ts = ts.iloc[1:,:]
    ts = ts.iloc[:37100,].values
    if df_tmp.loc[i, "split"] == "train" or df_tmp.loc[i, "split"] == "val":
        X_train.append(ts)
        y_train.append(df_tmp.loc[i, "anomaly_label_id"])
    else:
        X_test.append(ts)
        y_test.append(df_tmp.loc[i, "anomaly_label_id"])
        
X_train = np.stack(X_train, axis=0)
X_train = np.transpose(X_train, (0, 2, 1))
y_train = np.array(y_train)
X_test = np.stack(X_test, axis=0)
X_test = np.transpose(X_test, (0, 2, 1))
y_test = np.array(y_test)

[09/23/26 01:22:26] INFO     Loading data from training.df_splitted (CSVDataset)...            ]8;id=15629417;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=15629418;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\kedro\io\data_catalog.py#1050\1050]8;;\

In [5]:
target_names = [str(label) for label in all_labels]

results = {}

clfs = [
    SummaryClassifier(summary_functions=('mean', 'std', 'min', 'max'), summary_quantiles=(0.25, 0.5, 0.75), estimator=None, n_jobs=1, random_state=None),
    Catch22Classifier(outlier_norm=False, replace_nans=True, estimator=None, n_jobs=1, random_state=None),
    KNeighborsTimeSeriesClassifier(distance="euclidean"),
    CNNClassifierTorch(num_epochs=20, batch_size=16, verbose=True),
    ResNetClassifier(n_epochs=20, batch_size=16, verbose=True),
    InceptionTimeClassifierTorch(num_epochs=20, batch_size=16, verbose=True),
    LSTMFCNClassifierTorch(num_epochs=20, batch_size=16, verbose=True)
]


for clf in clfs:
    model_name = clf.__class__.__name__
    
    debug(f"Started training for {model_name}")
    
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    
    report = classification_report(
        y_true=y_test,
        y_pred=y_pred,
        target_names=target_names,
        digits=4,
        zero_division=0,
    )

    model_name = clf.__class__.__name__
    results[model_name] = report
    print(model_name)
    print(results[model_name])

    debug(f"Report Generated to: {model_name}")

for model_name, report in results.items():
    debug(model_name)
    debug(report)
    print()

[2026-09-12 23:24:38] Started training for SummaryClassifier
SummaryClassifier
              precision    recall  f1-score   support

        belt     1.0000    1.0000    1.0000        39
    magnetic     1.0000    1.0000    1.0000        37
      normal     1.0000    1.0000    1.0000        80

    accuracy                         1.0000       156
   macro avg     1.0000    1.0000    1.0000       156
weighted avg     1.0000    1.0000    1.0000       156

[2026-09-12 23:25:01] Report Generated to: SummaryClassifier
[2026-09-12 23:25:01] Started training for Catch22Classifier
Catch22Classifier
              precision    recall  f1-score   support

        belt     1.0000    1.0000    1.0000        39
    magnetic     1.0000    1.0000    1.0000        37
      normal     1.0000    1.0000    1.0000        80

    accuracy                         1.0000       156
   macro avg     1.0000    1.0000    1.0000       156
weighted avg     1.0000    1.0000    1.0000       156

[2026-09-12 23:27:0

[09/12/26 23:27:31] WARNING  TensorFlow GPU support is not available on native Windows for TensorFlow ]8;id=3298919;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\tensorflow\python\framework\config.py\config.py]8;;\:]8;id=3298920;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\tensorflow\python\framework\config.py#464\464]8;;\
                             >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please               
                             use WSL2 or the TensorFlow-DirectML plugin.                                           

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 37100, 3)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d (Conv1D)               │ (None, 37100, 64)         │           1,600 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 37100, 64)         │             256 │ conv1d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 37100, 64)         │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_1 (Conv1D)             │ (None, 37100, 64)         │          20,544 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 37100, 64)         │             256 │ conv1d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 37100, 64)         │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_3 (Conv1D)             │ (None, 37100, 64)         │             256 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_2 (Conv1D)             │ (None, 37100, 64)         │          12,352 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 37100, 64)         │             256 │ conv1d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 37100, 64)         │             256 │ conv1d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 37100, 64)         │               0 │ batch_normalization_3[0][… │
│                               │                           │                 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 37100, 64)         │               0 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_4 (Conv1D)             │ (None, 37100, 128)        │          65,664 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_4         │ (None, 37100, 128)        │             51

 Total params: 508,099 (1.94 MB)

 Trainable params: 505,539 (1.93 MB)

 Non-trainable params: 2,560 (10.00 KB)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 448s 44s/step - accuracy: 0.5641 - loss: 1.2242
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 434s 43s/step - accuracy: 0.5641 - loss: 0.8597
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 431s 43s/step - accuracy: 0.7244 - loss: 0.6940
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 445s 44s/step - accuracy: 0.7372 - loss: 0.5944
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 432s 43s/step - accuracy: 0.7436 - loss: 0.5283
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 429s 43s/step - accuracy: 0.7821 - loss: 0.4575
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 431s 43s/step - accuracy: 0.7821 - loss: 0.4636
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 438s 44s/step - accuracy: 0.8205 - loss: 0.4875
Epoch 9/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 434s 43s/step - accuracy: 0.7821 - loss: 0.4389
Epoch 10/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 433s 43s/step - accuracy: 0.8590 - loss: 0.4156
Epoch 11/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 432s 43s/step - accuracy: 0.8974 - loss: 0.3057
Epoch 12/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 443s 44s/step

[09/13/26 01:52:27] WARNING  C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packag ]8;id=3298925;file://C:\Users\tmdp1\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\warnings.py\warnings.py]8;;\:]8;id=3298926;file://C:\Users\tmdp1\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\warnings.py#112\112]8;;\
                             es\torch\nn\modules\conv.py:380: UserWarning: Using padding='same'                    
                             with even kernel lengths and odd dilation may require a zero-padded                   
                             copy of the input be created (Triggered internally at                                 
                             C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convoluti                
                             on.cpp:1102.)                                                                         
                               return F.conv1d(                                                                    
                                                                                                                   

Epoch 1: Loss: 0.8197458148002624
Epoch 2: Loss: 0.5008264392614364
Epoch 3: Loss: 0.32903375178575517
Epoch 4: Loss: 0.21623782366514205
Epoch 5: Loss: 0.1266644861549139
Epoch 6: Loss: 0.16150446832180024
Epoch 7: Loss: 0.21855515390634536
Epoch 8: Loss: 0.08626349903643131
Epoch 9: Loss: 0.07253220975399018
Epoch 10: Loss: 0.0329097987152636
Epoch 11: Loss: 0.02510950416326523
Epoch 12: Loss: 0.018837615428492428
Epoch 13: Loss: 0.015569336293265224
Epoch 14: Loss: 0.013923049671575427
Epoch 15: Loss: 0.010918855911586433
Epoch 16: Loss: 0.009119186038151383
Epoch 17: Loss: 0.00801059416262433
Epoch 18: Loss: 0.00707211647531949
Epoch 19: Loss: 0.006294713856186717
Epoch 20: Loss: 0.005743363127112389
InceptionTimeClassifierTorch
              precision    recall  f1-score   support

        belt     1.0000    1.0000    1.0000        39
    magnetic     1.0000    1.0000    1.0000        37
      normal     1.0000    1.0000    1.0000        80

    accuracy                         1.

# Train on Synthetic, Test on Real

In [4]:
samples_to_generate = {
    # belt
    ('spd1000', 'belt'): 5,
    ('spd1100', 'belt'): 5,
    ('spd1200', 'belt'): 4,
    ('spd1300', 'belt'): 5,
    ('spd1400', 'belt'): 4,
    ('spd1500', 'belt'): 2,
    ('spd1600', 'belt'): 2,
    ('spd1700', 'belt'): 2,
    ('spd1800', 'belt'): 2,
    ('spd1900', 'belt'): 3,
    ('spd2000', 'belt'): 2,
    ('spd2400', 'belt'): 2,
    ('spd2800', 'belt'): 2,
    ('spd3000', 'belt'): 2,

    # magnetic
    ('spd1000', 'magnetic'): 4,
    ('spd1100', 'magnetic'): 4,
    ('spd1200', 'magnetic'): 4,
    ('spd1300', 'magnetic'): 4,
    ('spd1400', 'magnetic'): 4,
    ('spd1500', 'magnetic'): 2,
    ('spd1600', 'magnetic'): 2,
    ('spd1700', 'magnetic'): 2,
    ('spd1800', 'magnetic'): 2,
    ('spd1900', 'magnetic'): 2,
    ('spd2000', 'magnetic'): 2,
    ('spd2400', 'magnetic'): 2,
    ('spd2800', 'magnetic'): 2,
    ('spd3000', 'magnetic'): 2,

    # normal
    ('spd1000', 'normal'): 8,
    ('spd1100', 'normal'): 8,
    ('spd1200', 'normal'): 8,
    ('spd1300', 'normal'): 8,
    ('spd1400', 'normal'): 8,
    ('spd1500', 'normal'): 4,
    ('spd1600', 'normal'): 4,
    ('spd1700', 'normal'): 4,
    ('spd1800', 'normal'): 4,
    ('spd1900', 'normal'): 4,
    ('spd2000', 'normal'): 4,
    ('spd2400', 'normal'): 4,
    ('spd2800', 'normal'): 4,
    ('spd3000', 'normal'): 4
}

In [5]:
syn_data_list = []
labels_spd = []
labels_anomaly_id = []

for spd, label in samples_to_generate.keys():
    generate_num = samples_to_generate[(spd, label)]
    synthetic_data_path = f"../data/07_model_output/synthetic_data_{spd}_{label}.npy"
    data = np.load(synthetic_data_path)
    data = data[:generate_num]
    data = data.transpose(0, 2, 1)
    syn_data_list.append(data)
    labels_spd.extend([spd] * generate_num)
    label_id = label_map[label]
    labels_anomaly_id.extend([label_id] * generate_num)

X_train_syn = np.concatenate(syn_data_list, axis=0)
y_train_syn = np.array(labels_anomaly_id)
print(f"Shape de X_train_syn: {X_train_syn.shape}")
print(f"Shape de y_train_syn: {y_train_syn.shape}")


from sktime.classification.feature_based import SummaryClassifier
from sktime.classification.feature_based import Catch22Classifier
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from sktime.classification.deep_learning.cnn import CNNClassifierTorch
from sktime.classification.deep_learning.resnet import ResNetClassifier
from sktime.classification.deep_learning.inceptiontime import InceptionTimeClassifierTorch
from sktime.classification.deep_learning.lstmfcn import LSTMFCNClassifierTorch

from sklearn.metrics import classification_report

target_names = [str(label) for label in all_labels]

results = {}

clfs = [
    SummaryClassifier(summary_functions=('mean', 'std', 'min', 'max'), summary_quantiles=(0.25, 0.5, 0.75), estimator=None, n_jobs=1, random_state=None),
    Catch22Classifier(outlier_norm=False, replace_nans=True, estimator=None, n_jobs=1, random_state=None),
    KNeighborsTimeSeriesClassifier(distance="euclidean"),
    CNNClassifierTorch(num_epochs=20, batch_size=16, verbose=True),
    ResNetClassifier(n_epochs=20, batch_size=16, verbose=True),
    InceptionTimeClassifierTorch(num_epochs=20, batch_size=16, verbose=True),
    LSTMFCNClassifierTorch(num_epochs=20, batch_size=16, verbose=True)
]


for clf in clfs:
    model_name = clf.__class__.__name__
    
    debug(f"Started training for {model_name}")
    
    clf.fit(X_train_syn, y_train_syn)
    y_pred = clf.predict(X_test)
    
    report = classification_report(
        y_true=y_test,
        y_pred=y_pred,
        target_names=target_names,
        digits=4,
        zero_division=0,
    )

    model_name = clf.__class__.__name__
    results[model_name] = report
    print(model_name)
    print(results[model_name])

    debug(f"Report Generated to: {model_name}")

for model_name, report in results.items():
    debug(model_name)
    debug(report)
    print()

Shape de X_train_syn: (156, 3, 37100)
Shape de y_train_syn: (156,)
[2026-09-23 01:25:06] Started training for SummaryClassifier
SummaryClassifier
              precision    recall  f1-score   support

        belt     0.9268    0.9744    0.9500        39
    magnetic     0.6452    0.5405    0.5882        37
      normal     0.8095    0.8500    0.8293        80

    accuracy                         0.8077       156
   macro avg     0.7938    0.7883    0.7892       156
weighted avg     0.7999    0.8077    0.8023       156

[2026-09-23 01:25:52] Report Generated to: SummaryClassifier
[2026-09-23 01:25:52] Started training for Catch22Classifier
Catch22Classifier
              precision    recall  f1-score   support

        belt     1.0000    1.0000    1.0000        39
    magnetic     0.8889    0.8649    0.8767        37
      normal     0.9383    0.9500    0.9441        80

    accuracy                         0.9423       156
   macro avg     0.9424    0.9383    0.9403       156
weighte

[09/23/26 01:30:03] WARNING  TensorFlow GPU support is not available on native Windows for TensorFlow ]8;id=15629425;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\tensorflow\python\framework\config.py\config.py]8;;\:]8;id=15629426;file://C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packages\tensorflow\python\framework\config.py#464\464]8;;\
                             >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please               
                             use WSL2 or the TensorFlow-DirectML plugin.                                           

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 37100, 3)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d (Conv1D)               │ (None, 37100, 64)         │           1,600 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization           │ (None, 37100, 64)         │             256 │ conv1d[0][0]               │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation (Activation)       │ (None, 37100, 64)         │               0 │ batch_normalization[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_1 (Conv1D)             │ (None, 37100, 64)         │          20,544 │ activation[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_1         │ (None, 37100, 64)         │             256 │ conv1d_1[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_1 (Activation)     │ (None, 37100, 64)         │               0 │ batch_normalization_1[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_3 (Conv1D)             │ (None, 37100, 64)         │             256 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_2 (Conv1D)             │ (None, 37100, 64)         │          12,352 │ activation_1[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_3         │ (None, 37100, 64)         │             256 │ conv1d_3[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_2         │ (None, 37100, 64)         │             256 │ conv1d_2[0][0]             │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add (Add)                     │ (None, 37100, 64)         │               0 │ batch_normalization_3[0][… │
│                               │                           │                 │ batch_normalization_2[0][… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_2 (Activation)     │ (None, 37100, 64)         │               0 │ add[0][0]                  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_4 (Conv1D)             │ (None, 37100, 128)        │          65,664 │ activation_2[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_4         │ (None, 37100, 128)        │             51

 Total params: 508,099 (1.94 MB)

 Trainable params: 505,539 (1.93 MB)

 Non-trainable params: 2,560 (10.00 KB)

Epoch 1/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 575s 55s/step - accuracy: 0.5321 - loss: 1.1518
Epoch 2/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 458s 45s/step - accuracy: 0.6859 - loss: 0.6554
Epoch 3/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 438s 43s/step - accuracy: 0.7244 - loss: 0.5776
Epoch 4/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 440s 44s/step - accuracy: 0.8013 - loss: 0.4583
Epoch 5/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 434s 43s/step - accuracy: 0.8718 - loss: 0.4235
Epoch 6/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 482s 48s/step - accuracy: 0.7885 - loss: 0.4549
Epoch 7/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 512s 51s/step - accuracy: 0.7500 - loss: 0.4721
Epoch 8/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 514s 52s/step - accuracy: 0.8141 - loss: 0.4010
Epoch 9/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 507s 50s/step - accuracy: 0.8782 - loss: 0.3640
Epoch 10/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 506s 50s/step - accuracy: 0.9295 - loss: 0.2804
Epoch 11/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 505s 50s/step - accuracy: 0.8654 - loss: 0.2830
Epoch 12/20
10/10 ━━━━━━━━━━━━━━━━━━━━ 511s 51s/step

[09/23/26 04:17:47] WARNING  C:\Users\tmdp1\masters-dissertation-data-science\.venv\Lib\site-packag ]8;id=15629431;file://C:\Users\tmdp1\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\warnings.py\warnings.py]8;;\:]8;id=15629432;file://C:\Users\tmdp1\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\warnings.py#112\112]8;;\
                             es\torch\nn\modules\conv.py:380: UserWarning: Using padding='same'                    
                             with even kernel lengths and odd dilation may require a zero-padded                   
                             copy of the input be created (Triggered internally at                                 
                             C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convoluti                
                             on.cpp:1102.)                                                                         
                               return F.conv1d(                                                                    
                                                                                                                   

Epoch 1: Loss: 1.210086178779602
Epoch 2: Loss: 1.1035890758037568
Epoch 3: Loss: 1.0227960348129272
Epoch 4: Loss: 1.0015677452087401
Epoch 5: Loss: 0.9934316098690033
Epoch 6: Loss: 0.9881118953227996
Epoch 7: Loss: 0.9808730840682983
Epoch 8: Loss: 0.9695504397153855
Epoch 9: Loss: 0.9600898027420044
Epoch 10: Loss: 0.9617277026176453
Epoch 11: Loss: 0.9601852297782898
Epoch 12: Loss: 0.9652592450380325
Epoch 13: Loss: 0.9703518077731133
Epoch 14: Loss: 0.9698630750179291
Epoch 15: Loss: 0.948744997382164
Epoch 16: Loss: 0.9393634170293808
Epoch 17: Loss: 0.9336884200572968
Epoch 18: Loss: 0.9285678863525391
Epoch 19: Loss: 0.9240904912352562
Epoch 20: Loss: 0.9290468737483024
InceptionTimeClassifierTorch
              precision    recall  f1-score   support

        belt     0.0000    0.0000    0.0000        39
    magnetic     0.4143    0.7838    0.5421        37
      normal     0.4535    0.4875    0.4699        80

    accuracy                         0.4359       156
   macro a

# Using much more samples

In [ ]:
syn_data_list = []
labels_spd = []
labels_anomaly_id = []

for spd, label in samples_to_generate.keys():
    # generate_num = samples_to_generate[(spd, label)]
    generate_num = 10
    synthetic_data_path = f"../data/07_model_output/synthetic_data_{spd}_{label}.npy"
    data = np.load(synthetic_data_path)
    data = data[:generate_num]
    data = data.transpose(0, 2, 1)
    syn_data_list.append(data)
    labels_spd.extend([spd] * generate_num)
    label_id = label_map[label]
    labels_anomaly_id.extend([label_id] * generate_num)

X_train_syn = np.concatenate(syn_data_list, axis=0)
y_train_syn = np.array(labels_anomaly_id)
print(f"Shape de X_train_syn: {X_train_syn.shape}")
print(f"Shape de y_train_syn: {y_train_syn.shape}")


from sktime.classification.feature_based import SummaryClassifier
from sktime.classification.feature_based import Catch22Classifier
from sktime.classification.distance_based import KNeighborsTimeSeriesClassifier
from sktime.classification.deep_learning.cnn import CNNClassifierTorch
from sktime.classification.deep_learning.resnet import ResNetClassifier
from sktime.classification.deep_learning.inceptiontime import InceptionTimeClassifierTorch
from sktime.classification.deep_learning.lstmfcn import LSTMFCNClassifierTorch

from sklearn.metrics import classification_report

target_names = [str(label) for label in all_labels]

results = {}

clfs = [
    SummaryClassifier(summary_functions=('mean', 'std', 'min', 'max'), summary_quantiles=(0.25, 0.5, 0.75), estimator=None, n_jobs=1, random_state=None),
    Catch22Classifier(outlier_norm=False, replace_nans=True, estimator=None, n_jobs=1, random_state=None),
    KNeighborsTimeSeriesClassifier(distance="euclidean"),
    CNNClassifierTorch(num_epochs=20, batch_size=16, verbose=True),
    ResNetClassifier(n_epochs=20, batch_size=16, verbose=True),
    InceptionTimeClassifierTorch(num_epochs=20, batch_size=16, verbose=True),
    LSTMFCNClassifierTorch(num_epochs=20, batch_size=16, verbose=True)
]


for clf in clfs:
    model_name = clf.__class__.__name__
    
    debug(f"Started training for {model_name}")
    
    clf.fit(X_train_syn, y_train_syn)
    y_pred = clf.predict(X_test)
    
    report = classification_report(
        y_true=y_test,
        y_pred=y_pred,
        target_names=target_names,
        digits=4,
        zero_division=0,
    )

    model_name = clf.__class__.__name__
    results[model_name] = report
    print(model_name)
    print(results[model_name])

    debug(f"Report Generated to: {model_name}")

for model_name, report in results.items():
    debug(model_name)
    debug(report)
    print()

Shape de X_train_syn: (420, 3, 37100)
Shape de y_train_syn: (420,)
[2026-09-23 07:59:48] Started training for SummaryClassifier
SummaryClassifier
              precision    recall  f1-score   support

        belt     0.9500    0.9744    0.9620        39
    magnetic     0.4688    0.8108    0.5941        37
      normal     0.8654    0.5625    0.6818        80

    accuracy                         0.7244       156
   macro avg     0.7614    0.7826    0.7460       156
weighted avg     0.7925    0.7244    0.7311       156

[2026-09-23 08:00:29] Report Generated to: SummaryClassifier
[2026-09-23 08:00:29] Started training for Catch22Classifier
Catch22Classifier
              precision    recall  f1-score   support

        belt     1.0000    1.0000    1.0000        39
    magnetic     0.8571    0.9730    0.9114        37
      normal     0.9867    0.9250    0.9548        80

    accuracy                         0.9551       156
   macro avg     0.9479    0.9660    0.9554       156
weighte

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)    │ (None, 37100, 3)          │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_11 (Conv1D)            │ (None, 37100, 64)         │           1,600 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_12        │ (None, 37100, 64)         │             256 │ conv1d_11[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_9 (Activation)     │ (None, 37100, 64)         │               0 │ batch_normalization_12[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_12 (Conv1D)            │ (None, 37100, 64)         │          20,544 │ activation_9[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_13        │ (None, 37100, 64)         │             256 │ conv1d_12[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_10 (Activation)    │ (None, 37100, 64)         │               0 │ batch_normalization_13[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_14 (Conv1D)            │ (None, 37100, 64)         │             256 │ input_layer_1[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_13 (Conv1D)            │ (None, 37100, 64)         │          12,352 │ activation_10[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_15        │ (None, 37100, 64)         │             256 │ conv1d_14[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_14        │ (None, 37100, 64)         │             256 │ conv1d_13[0][0]            │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ add_3 (Add)                   │ (None, 37100, 64)         │               0 │ batch_normalization_15[0]… │
│                               │                           │                 │ batch_normalization_14[0]… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_11 (Activation)    │ (None, 37100, 64)         │               0 │ add_3[0][0]                │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_15 (Conv1D)            │ (None, 37100, 128)        │          65,664 │ activation_11[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ batch_normalization_16        │ (None, 37100, 128)        │             51

 Total params: 508,099 (1.94 MB)

 Trainable params: 505,539 (1.93 MB)

 Non-trainable params: 2,560 (10.00 KB)

Epoch 1/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1391s 50s/step - accuracy: 0.5952 - loss: 0.9201
Epoch 2/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1373s 51s/step - accuracy: 0.7333 - loss: 0.5405
Epoch 3/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1184s 44s/step - accuracy: 0.8429 - loss: 0.3842
Epoch 4/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1214s 45s/step - accuracy: 0.9095 - loss: 0.2863
Epoch 5/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1329s 49s/step - accuracy: 0.8667 - loss: 0.3876
Epoch 6/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1271s 47s/step - accuracy: 0.9095 - loss: 0.2509
Epoch 7/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1441s 53s/step - accuracy: 0.9310 - loss: 0.1715
Epoch 8/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1248s 46s/step - accuracy: 0.9429 - loss: 0.1568
Epoch 9/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1276s 47s/step - accuracy: 0.9643 - loss: 0.1184
Epoch 10/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1263s 47s/step - accuracy: 0.9619 - loss: 0.0959
Epoch 11/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 1357s 50s/step - accuracy: 0.9643 - loss: 0.0967
Epoch 12/20
27/27 ━━━━━━━━━━━━━━━━━━━━ 12